In [18]:
# Cell 1 — imports and connection
import os
import pandas as pd
from dotenv import load_dotenv
import snowflake.connector
from collections import Counter
import json
 
load_dotenv()
 
conn = snowflake.connector.connect(
    user=os.getenv('SNOWFLAKE_USER'),
    password=os.getenv('SNOWFLAKE_PASSWORD'),
    account=os.getenv('SNOWFLAKE_ACCOUNT'),
    role=os.environ["SNOWFLAKE_ROLE"],
    warehouse=os.getenv('SNOWFLAKE_WAREHOUSE'),
    database='ANALYTICS_PROD',
    schema='PUBLIC',
)
 
def run_query(sql: str) -> pd.DataFrame:
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [desc[0] for desc in cur.description]
    cur.close()
    return pd.DataFrame(rows, columns=cols)
 
print("Connected.")

Connected.


In [2]:
# ── Cell 2 — load mart ────────────────────────────────────────────────────────
df = run_query("SELECT * FROM ANALYTICS_PROD.PUBLIC.FCT_JOB_POSTINGS")
df.columns = [c.lower() for c in df.columns]
 
# Parse arrays
def parse_arr(val):
    if isinstance(val, list): return val
    if isinstance(val, str):
        try: return json.loads(val)
        except: return []
    return []
 
for col in ["tech_stack_required", "tech_stack_preferred", "paradigms_required", "paradigms_preferred"]:
    df[col] = df[col].apply(parse_arr)
 
# Parse dates
for col in ["date_posted", "ingested_at", "enriched_at"]:
    df[col] = pd.to_datetime(df[col], errors="coerce")
 
# Numeric
for col in ["final_salary_min", "final_salary_max", "years_required_min", "years_required_max", "confidence_score"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
 
# Booleans
for col in ["acknowledges_ai", "explicitly_encourages_applicants", "is_explicitly_entry_level"]:
    df[col] = df[col].apply(lambda x: True if str(x).strip().upper() in ("TRUE", "1", "YES") else False)
 
# Effective seniority
SENIORITY_ORDER = ["entry_level", "junior", "mid_level"]
df["effective_seniority"] = df["listed_seniority"].where(
    df["listed_seniority"].isin(SENIORITY_ORDER),
    other=df["is_explicitly_entry_level"].map({True: "entry_level", False: None})
)
 
# Salary mid
salary_df = df.dropna(subset=["final_salary_min", "final_salary_max"]).copy()
salary_df["salary_mid"] = (salary_df["final_salary_min"] + salary_df["final_salary_max"]) / 2
 
print(f"Loaded {len(df)} postings")

Loaded 210 postings


In [3]:
# ── Cell 3 — top level snapshot ───────────────────────────────────────────────
print("=== TOP LEVEL ===")
print(f"Total postings:       {len(df)}")
print(f"Unique companies:     {df['company_name'].nunique()}")
print(f"Role types:           {df['ingestion_query'].nunique()}")
print(f"Sources:              {df['source'].nunique()}")
print(f"Salary disclosed:     {len(salary_df)} ({len(salary_df)/len(df):.0%})")
print(f"LLM enriched:         {df['role_archetype'].notna().sum()} ({df['role_archetype'].notna().mean():.0%})")
print(f"Date range:           {df['date_posted'].min().date()} → {df['date_posted'].max().date()}")
print(f"Last ingested:        {df['ingested_at'].max().date()}")
 

=== TOP LEVEL ===
Total postings:       210
Unique companies:     177
Role types:           3
Sources:              3
Salary disclosed:     122 (58%)
LLM enriched:         208 (99%)
Date range:           2026-05-20 → 2026-06-15
Last ingested:        2026-06-15


In [4]:
# ── Cell 4 — postings by role type ───────────────────────────────────────────
print("\n=== POSTINGS BY ROLE TYPE ===")
print(df["ingestion_query"].value_counts().to_string())


=== POSTINGS BY ROLE TYPE ===
ingestion_query
Data Analyst          149
Data Engineer          35
Analytics Engineer     26


In [5]:
# ── Cell 5 — postings by source ──────────────────────────────────────────────
print("\n=== POSTINGS BY SOURCE ===")
print(df["source"].value_counts().to_string())
 
print("\n=== POSTINGS BY ROLE × SOURCE ===")
print(df.groupby(["ingestion_query", "source"]).size().unstack(fill_value=0).to_string())


=== POSTINGS BY SOURCE ===
source
jsearch       89
theirstack    68
builtin       53

=== POSTINGS BY ROLE × SOURCE ===
source              builtin  jsearch  theirstack
ingestion_query                                 
Analytics Engineer        1       20           5
Data Analyst             39       65          45
Data Engineer            13        4          18


In [6]:
# ── Cell 6 — work model ───────────────────────────────────────────────────────
print("\n=== WORK MODEL (overall) ===")
print(df["work_model"].value_counts().to_string())
 
print("\n=== WORK MODEL by ROLE ===")
print(df.groupby(["ingestion_query", "work_model"]).size().unstack(fill_value=0).to_string())


=== WORK MODEL (overall) ===
work_model
onsite    137
remote     51
hybrid     22

=== WORK MODEL by ROLE ===
work_model          hybrid  onsite  remote
ingestion_query                           
Analytics Engineer       2      21       3
Data Analyst            14     103      32
Data Engineer            6      13      16


In [7]:
# ── Cell 7 — seniority distribution ──────────────────────────────────────────
print("\n=== EFFECTIVE SENIORITY (overall) ===")
print(df["effective_seniority"].value_counts().to_string())
 
print("\n=== EFFECTIVE SENIORITY by ROLE ===")
print(df.groupby(["ingestion_query", "effective_seniority"]).size().unstack(fill_value=0).to_string())


=== EFFECTIVE SENIORITY (overall) ===
effective_seniority
mid_level      83
junior         23
entry_level    10

=== EFFECTIVE SENIORITY by ROLE ===
effective_seniority  entry_level  junior  mid_level
ingestion_query                                    
Analytics Engineer             2       0          6
Data Analyst                   7      19         51
Data Engineer                  1       4         26


In [9]:
# ── Cell 8 — salary by role ───────────────────────────────────────────────────
print("\n=== SALARY by ROLE (median, where disclosed) ===")
sal_by_role = (
    salary_df.groupby("ingestion_query")["salary_mid"]
    .agg(["median", "count", "min", "max", "std"])
    .round(0)
)
sal_by_role.columns = ["median", "n", "min", "max", "std"]
print(sal_by_role.to_string())
 
print("\n=== SALARY by ROLE × EFFECTIVE SENIORITY ===")
sal_sen = salary_df[salary_df["effective_seniority"].isin(SENIORITY_ORDER)]
print(
    sal_sen.groupby(["ingestion_query", "effective_seniority"])["salary_mid"]
    .agg(["median", "count"])
    .round(0)
    .to_string()
)


=== SALARY by ROLE (median, where disclosed) ===
                      median   n       min       max      std
ingestion_query                                              
Analytics Engineer  142375.0  18   55000.0  445000.0  86232.0
Data Analyst         92500.0  86    2080.0  322500.0  38220.0
Data Engineer       150825.0  18  102500.0  202500.0  36164.0

=== SALARY by ROLE × EFFECTIVE SENIORITY ===
                                          median  count
ingestion_query    effective_seniority                 
Analytics Engineer entry_level           89750.0      2
                   mid_level            135000.0      5
Data Analyst       entry_level          112250.0      4
                   junior                92500.0     11
                   mid_level             93750.0     28
Data Engineer      entry_level          120000.0      1
                   junior               112500.0      3
                   mid_level            160000.0     13


In [10]:
# ── Cell 9 — AI blindspot ─────────────────────────────────────────────────────
print("\n=== AI ACKNOWLEDGMENT (overall) ===")
print(f"Acknowledges AI: {df['acknowledges_ai'].sum()} of {len(df)} ({df['acknowledges_ai'].mean():.0%})")
 
print("\n=== AI ACKNOWLEDGMENT by ROLE ===")
ai = (
    df.groupby("ingestion_query")["acknowledges_ai"]
    .agg(["sum", "count", "mean"])
    .round(3)
)
ai.columns = ["yes", "total", "rate"]
print(ai.to_string())


=== AI ACKNOWLEDGMENT (overall) ===
Acknowledges AI: 67 of 210 (32%)

=== AI ACKNOWLEDGMENT by ROLE ===
                    yes  total   rate
ingestion_query                      
Analytics Engineer   15     26  0.577
Data Analyst         37    149  0.248
Data Engineer        15     35  0.429


In [11]:
# ── Cell 10 — title vs archetype confusion ────────────────────────────────────
print("\n=== TITLE vs LLM ARCHETYPE (confusion matrix, counts) ===")
matrix_df = df.dropna(subset=["role_archetype"]).copy()
pivot = (
    matrix_df.groupby(["ingestion_query", "role_archetype"])
    .size()
    .unstack(fill_value=0)
)
print(pivot.to_string())
 
print("\n=== TITLE vs LLM ARCHETYPE (row %, agreement on diagonal) ===")
print(pivot.div(pivot.sum(axis=1), axis=0).round(2).to_string())
 
# Agreement rate
def queries_match(row):
    q = row["ingestion_query"].lower().replace(" ", "_").replace("-", "_")
    a = row["role_archetype"].lower()
    return bool(set(q.split("_")) & set(a.split("_")))
 
agree_n = matrix_df.apply(queries_match, axis=1).sum()
print(f"\nAgreement: {agree_n} of {len(matrix_df)} ({agree_n/len(matrix_df):.0%})")


=== TITLE vs LLM ARCHETYPE (confusion matrix, counts) ===
role_archetype      analytics_engineer  data_analyst  data_engineer  hybrid
ingestion_query                                                            
Analytics Engineer                   9             1             15       1
Data Analyst                         0           134              2      12
Data Engineer                        0             0             33       1

=== TITLE vs LLM ARCHETYPE (row %, agreement on diagonal) ===
role_archetype      analytics_engineer  data_analyst  data_engineer  hybrid
ingestion_query                                                            
Analytics Engineer                0.35          0.04           0.58    0.04
Data Analyst                      0.00          0.91           0.01    0.08
Data Engineer                     0.00          0.00           0.97    0.03

Agreement: 193 of 208 (93%)


In [12]:
# ── Cell 11 — top skills overall and by role ──────────────────────────────────
print("\n=== TOP 20 REQUIRED SKILLS (overall) ===")
all_req = [t for row in df["tech_stack_required"] if isinstance(row, list) for t in row if isinstance(t, str)]
print(pd.Series(Counter(all_req)).sort_values(ascending=False).head(20).to_string())
 
print("\n=== TOP 10 REQUIRED SKILLS by ROLE ===")
for query in df["ingestion_query"].unique():
    subset = df[df["ingestion_query"] == query]
    tools = [t for row in subset["tech_stack_required"] if isinstance(row, list) for t in row if isinstance(t, str)]
    top = pd.Series(Counter(tools)).sort_values(ascending=False).head(10)
    print(f"\n--- {query} ---")
    print(top.to_string())


=== TOP 20 REQUIRED SKILLS (overall) ===
sql           121
python         81
excel          62
tableau        27
r              25
power bi       23
snowflake      19
databricks     16
dbt            15
aws            14
powerpoint     13
airflow        13
oracle         12
word           10
azure           9
looker          9
spark           8
postgresql      7
pyspark         7
bigquery        7

=== TOP 10 REQUIRED SKILLS by ROLE ===

--- Data Analyst ---
sql           82
excel         60
python        40
tableau       23
r             23
power bi      22
powerpoint    13
word          10
looker         9
snowflake      9

--- Analytics Engineer ---
sql           19
python        17
airflow        6
snowflake      5
oracle         5
aws            4
databricks     4
dbt            3
nosql          3
redshift       3

--- Data Engineer ---
python        24
sql           20
databricks    10
aws            8
azure          8
airflow        7
dbt            6
snowflake      5
mysql    

In [13]:
# ── Cell 12 — top paradigms by role ──────────────────────────────────────────
print("\n=== TOP 10 PARADIGMS by ROLE ===")
for query in df["ingestion_query"].unique():
    subset = df[df["ingestion_query"] == query]
    paras = [
        t
        for req, pref in zip(subset["paradigms_required"], subset["paradigms_preferred"])
        for row in [req, pref] if isinstance(row, list)
        for t in row if isinstance(t, str)
    ]
    top = pd.Series(Counter(paras)).sort_values(ascending=False).head(10)
    print(f"\n--- {query} ---")
    print(top.to_string())


=== TOP 10 PARADIGMS by ROLE ===

--- Data Analyst ---
data analysis           52
data quality            34
data visualization      31
statistical analysis    26
data governance         24
data modeling           20
data validation         19
predictive modeling      9
data management          8
reporting                7

--- Analytics Engineer ---
data modeling       16
etl design          14
data quality        12
data governance     10
data warehousing     8
ci/cd                6
data testing         2
data mining          2
devops               2
data engineering     2

--- Data Engineer ---
etl design                23
data modeling             17
data warehousing          15
data quality              13
data governance           10
data integration           5
ci/cd                      4
pipeline orchestration     4
observability              3
data ingestion             3


In [14]:
# ── Cell 13 — experience requirements ────────────────────────────────────────
print("\n=== YEARS REQUIRED by ROLE (median, where specified) ===")
yrs = df.dropna(subset=["years_required_min"])
print(
    yrs.groupby("ingestion_query")["years_required_min"]
    .agg(["median", "count"])
    .round(1)
    .to_string()
)
 
print("\n=== YEARS REQUIRED by ROLE × EFFECTIVE SENIORITY ===")
yrs_sen = yrs[yrs["effective_seniority"].isin(SENIORITY_ORDER)]
print(
    yrs_sen.groupby(["ingestion_query", "effective_seniority"])["years_required_min"]
    .agg(["median", "count"])
    .round(1)
    .to_string()
)


=== YEARS REQUIRED by ROLE (median, where specified) ===
                    median  count
ingestion_query                  
Analytics Engineer     3.0     22
Data Analyst           3.0    115
Data Engineer          3.0     31

=== YEARS REQUIRED by ROLE × EFFECTIVE SENIORITY ===
                                        median  count
ingestion_query    effective_seniority               
Analytics Engineer entry_level             0.0      1
                   mid_level               2.0      6
Data Analyst       entry_level             1.0      2
                   junior                  2.0     13
                   mid_level               2.0     40
Data Engineer      junior                  1.0      4
                   mid_level               3.0     24


In [15]:
# ── Cell 14 — degree requirements ────────────────────────────────────────────
print("\n=== DEGREE REQUIREMENTS by ROLE ===")
deg = df.dropna(subset=["degree_requirement"])
print(
    deg.groupby(["ingestion_query", "degree_requirement"])
    .size()
    .unstack(fill_value=0)
    .to_string()
)


=== DEGREE REQUIREMENTS by ROLE ===
degree_requirement  bachelors  equivalent_ok  masters  none
ingestion_query                                            
Analytics Engineer          9              2        1    14
Data Analyst               87             13        5    43
Data Engineer              17              0        1    16


In [16]:
# ── Cell 15 — encourages applicants ──────────────────────────────────────────
print("\n=== ENCOURAGES APPLICANTS by ROLE ===")
enc = (
    df.groupby("ingestion_query")["explicitly_encourages_applicants"]
    .agg(["sum", "count", "mean"])
    .round(3)
)
enc.columns = ["yes", "total", "rate"]
print(enc.to_string())


=== ENCOURAGES APPLICANTS by ROLE ===
                    yes  total   rate
ingestion_query                      
Analytics Engineer    6     26  0.231
Data Analyst         42    149  0.282
Data Engineer         4     35  0.114


In [20]:
# Cell: before/after rate from the mart
df = run_query("""
    SELECT
        COUNT(*) AS total,
        SUM(CASE WHEN explicitly_encourages_applicants THEN 1 ELSE 0 END) AS true_count,
        ROUND(SUM(CASE WHEN explicitly_encourages_applicants THEN 1 ELSE 0 END) / COUNT(*)::FLOAT * 100, 1) AS true_pct
    FROM ANALYTICS_PROD.PUBLIC.FCT_JOB_POSTINGS
""")
print(df)

   TOTAL  TRUE_COUNT  TRUE_PCT
0    280          71      25.4


In [25]:
# Cell: spot-check what's actually triggering true
df = run_query("""
    SELECT job_id, job_title, ingestion_query, description
    FROM ANALYTICS_PROD.PUBLIC.FCT_JOB_POSTINGS
    WHERE explicitly_encourages_applicants = TRUE
    LIMIT 20
""")
df.columns = [c.lower() for c in df.columns]

for _, row in df.iterrows():
    desc = row['description'] or ""
    for keyword in ['encourage', 'apply anyway', "don't meet", 'do not meet', 'every qualification', 'every requirement', 'all requirements']:
        idx = desc.lower().find(keyword)
        if idx != -1:
            print(f"--- {row['job_title']} ({row['ingestion_query']}) ---")
            print(desc[max(0, idx-100):idx+150])
            print()
            break
    else:
        print(f"--- {row['job_title']} ({row['ingestion_query']}) — NO KEYWORD FOUND, check manually ---")
        print()

--- [P] Data Scientist, Policy (Data Analyst) ---
ery reasonable effort to get you a visa, and we retain an immigration lawyer to help with this.

We encourage you to apply even if you do not believe you meet every single qualification. Not all strong candidates will meet every single qualification 

--- Aladdin Integration Data Analyst (Data Analyst) — NO KEYWORD FOUND, check manually ---

--- Analytics Engineer - Music Creator Growth (Analytics Engineer) — NO KEYWORD FOUND, check manually ---

--- Analytics Engineer (Data Analyst) — NO KEYWORD FOUND, check manually ---

--- Analytics Engineer (Analytics Engineer) — NO KEYWORD FOUND, check manually ---

--- Analytics Engineer, Music Creator Growth (Analytics Engineer) — NO KEYWORD FOUND, check manually ---

--- Analytics Engineer, Service Ops Analytics & AI (Analytics Engineer) ---
If you are passionate about data engineering and ready to take on a challenging, impactful role, we encourage you to apply. Join us in building the next-ge

In [26]:
import re

df = run_query("""
    SELECT job_id, job_title, ingestion_query, description
    FROM ANALYTICS_PROD.PUBLIC.FCT_JOB_POSTINGS
    WHERE explicitly_encourages_applicants = TRUE
    LIMIT 20
""")
df.columns = [c.lower() for c in df.columns]

encourage_words = ['encourage', 'invite', 'still want to hear', 'love to hear']

# Use flexible regex patterns instead of exact phrases — allow optional words in between
qualification_patterns = [
    r'every\s+(\w+\s+)?(qualification|requirement|criteri[ao]|box)',
    r'all\s+(\w+\s+)?(requirement|qualification)',
    r"don'?t\s+meet",
    r'do\s+not\s+meet',
    r"doesn'?t\s+(align|match)",
    r"don'?t\s+match",
    r'not\s+meet\s+all',
    r'minimum\s+qualification',
    r'check(s)?\s+every\s+box',
]
qualification_re = re.compile('|'.join(qualification_patterns), re.IGNORECASE)

def find_real_signal(desc: str):
    desc_lower = desc.lower()
    for ew in encourage_words:
        for m in re.finditer(re.escape(ew), desc_lower):
            window_start = max(0, m.start() - 150)
            window_end = min(len(desc_lower), m.end() + 150)
            window = desc[window_start:window_end]  # use original case for display
            if qualification_re.search(window):
                return window
    return None

for _, row in df.iterrows():
    desc = row['description'] or ""
    hit = find_real_signal(desc)
    if hit:
        print(f"✅ REAL SIGNAL — {row['job_title']} ({row['ingestion_query']})")
        print(hit)
    else:
        print(f"❌ NO REAL SIGNAL FOUND — {row['job_title']} ({row['ingestion_query']}) — flagged TRUE by model anyway")
    print()

✅ REAL SIGNAL — [P] Data Scientist, Policy (Data Analyst)
date. But if we make you an offer, we will make every reasonable effort to get you a visa, and we retain an immigration lawyer to help with this.

We encourage you to apply even if you do not believe you meet every single qualification. Not all strong candidates will meet every single qualification as listed

❌ NO REAL SIGNAL FOUND — Aladdin Integration Data Analyst (Data Analyst) — flagged TRUE by model anyway

❌ NO REAL SIGNAL FOUND — Analytics Engineer - Music Creator Growth (Analytics Engineer) — flagged TRUE by model anyway

❌ NO REAL SIGNAL FOUND — Analytics Engineer (Data Analyst) — flagged TRUE by model anyway

❌ NO REAL SIGNAL FOUND — Analytics Engineer (Analytics Engineer) — flagged TRUE by model anyway

❌ NO REAL SIGNAL FOUND — Analytics Engineer, Music Creator Growth (Analytics Engineer) — flagged TRUE by model anyway

❌ NO REAL SIGNAL FOUND — Analytics Engineer, Service Ops Analytics & AI (Analytics Engineer) — flag

In [27]:
import re

# Pull a different batch — skip the first 20 we already looked at, get the next 20
df = run_query("""
    SELECT job_id, job_title, ingestion_query, description
    FROM ANALYTICS_PROD.PUBLIC.FCT_JOB_POSTINGS
    WHERE explicitly_encourages_applicants = TRUE
    LIMIT 20 OFFSET 20
""")
df.columns = [c.lower() for c in df.columns]

encourage_words = ['encourage', 'invite', 'still want to hear', 'love to hear']

qualification_patterns = [
    r'every\s+(\w+\s+)?(qualification|requirement|criteri[ao]|box)',
    r'all\s+(\w+\s+)?(requirement|qualification)',
    r"don'?t\s+meet",
    r'do\s+not\s+meet',
    r"doesn'?t\s+(align|match)",
    r"don'?t\s+match",
    r'not\s+meet\s+all',
    r'minimum\s+qualification',
    r'check(s)?\s+every\s+box',
]
qualification_re = re.compile('|'.join(qualification_patterns), re.IGNORECASE)

def find_windows(desc: str):
    """Return all encourage-word windows, flagging whether each one has real signal."""
    desc_lower = desc.lower()
    results = []
    for ew in encourage_words:
        for m in re.finditer(re.escape(ew), desc_lower):
            window_start = max(0, m.start() - 150)
            window_end = min(len(desc_lower), m.end() + 150)
            window = desc[window_start:window_end]
            has_signal = bool(qualification_re.search(window))
            results.append((has_signal, window))
    return results

for _, row in df.iterrows():
    desc = row['description'] or ""
    windows = find_windows(desc)
    if not windows:
        print(f"⚪ NO ENCOURAGE WORD FOUND AT ALL — {row['job_title']} ({row['ingestion_query']})")
    else:
        any_signal = any(has_signal for has_signal, _ in windows)
        marker = "✅ REAL SIGNAL" if any_signal else "❌ NO REAL SIGNAL"
        print(f"{marker} — {row['job_title']} ({row['ingestion_query']})")
        for has_signal, window in windows:
            tag = "  [MATCH]" if has_signal else "  [no qualifier nearby]"
            print(f"{tag} {window}")
    print()

⚪ NO ENCOURAGE WORD FOUND AT ALL — Data Analyst - Global Growth Luxury Strategy (Data Analyst)

❌ NO REAL SIGNAL — Data Analyst - Immediate Opening (Data Analyst)
  [no qualifier nearby] lowed, you directly impact the operational speed and safety for millions of people across the NY region.

Reliable and detail-oriented candidates are encouraged to apply. You must pass all pre-employment federal background checks in NY. While no prior security experience is required, a professional demeanor,

❌ NO REAL SIGNAL — Data Analyst - Model Credibility (Data Analyst)
  [no qualifier nearby] ol of Medicine is an equal opportunity employer and committed to inclusion in all aspects of recruiting and employment. All qualified individuals are encouraged to apply and will receive consideration. We require applications to be completed online.
  
  
View Know Your Rights: Workplace discrimination is il

❌ NO REAL SIGNAL — Data Analyst (Data Analyst)
  [no qualifier nearby] d so much more. adMarketplace